In [2]:
from __future__ import annotations

import ad_hoc_diffractometer as ahd
from ad_hoc_diffractometer.stage import Stage

In [3]:
BASIS = ahd.factories.BASIS_BL
TRANSVERSE = BASIS["transverse"]  # +x
LONGITUDINAL = BASIS["longitudinal"]  # +y
VERTICAL = BASIS["vertical"]  # +z

In [8]:
stages = [
    Stage("omega", -TRANSVERSE, parent=None, role="sample"),
    Stage("chi", +LONGITUDINAL, parent="omega", role="sample"),
    Stage("phi", -TRANSVERSE, parent="chi", role="sample"),
    Stage("ttheta", -TRANSVERSE, parent=None, role="detector"),
]

In [9]:
modes = {
    "bisecting": ahd.ConstraintSet(
        [ahd.BisectConstraint("omega", "ttheta")],
        computed=["omega", "chi", "phi", "ttheta"],
    ),
    "fixed_chi": ahd.ConstraintSet(
        [ahd.SampleConstraint("chi", 90.0)],
        computed=["omega", "phi", "ttheta"],
    ),
    "fixed_phi": ahd.ConstraintSet(
        [ahd.SampleConstraint("phi", 0.0)],
        computed=["omega", "chi", "ttheta"],
    ),
}

In [12]:
@ahd.register_geometry
def my_fourcv(**kwargs):
    g = ahd.AdHocDiffractometer(
        name="my_fourcv",
        stages=stages,
        basis=BASIS,
        description="Four-circle Eulerian, vertical scattering plane",
        modes=modes,
        default_mode="bisecting",
    )
    return g

In [14]:
g.wavelength = 1.5406
g.sample.lattice = ahd.Lattice(a=5.431)

NameError: name 'g' is not defined

In [ ]:
print(g.summary())

AdHocDiffractometer: my_fourcv
  Four-circle Eulerian, vertical scattering plane
  λ = 1.540600 Å  (E = 8.047786 keV)

  Basis vectors:
    vertical     -> +z
    longitudinal -> +y
    transverse   -> +x

  Sample stages (floor first):
    omega     axis=-x    parent=None  angle=0.0 deg
    chi       axis=+y    parent=omega  angle=0.0 deg
    phi       axis=-x    parent=chi  angle=0.0 deg

  Detector stages (floor first):
    ttheta    axis=-x    parent=None  angle=0.0 deg
None


In [ ]:
ahd.ub_identity(g.sample)

array([[ 0.00000000e+00,  0.00000000e+00,  1.15691131e+00],
       [ 1.15691131e+00, -7.08403864e-17, -7.08403864e-17],
       [ 0.00000000e+00,  1.15691131e+00, -7.08403864e-17]])

In [ ]:
solutions = g.forward(0, 0, 4)
for sol in solutions:
    print(sol)

{'omega': 34.564641808171025, 'chi': 90.0, 'phi': 9.532985381462646e-11, 'ttheta': 69.12928361634205}
{'omega': 34.564641808171025, 'chi': 90.0, 'phi': 179.9999999967004, 'ttheta': 69.12928361634205}
{'omega': 34.564641808171025, 'chi': 90.0, 'phi': 90.00000000009534, 'ttheta': 69.12928361634205}


In [ ]:
## Concise form
g2 = ahd.make_geometry("fourcv")
g2.wavelength = 1.5406
g2.sample.lattice = ahd.Lattice(a=5.431)
print(g2.summary())

AdHocDiffractometer: fourcv
  Eulerian Four-Circle (vertical).

Busing & Levy (1967) four-circle Eulerian diffractometer; vertical
scattering plane, transverse ttheta, synchrotron convention.


Walko (2016) designation: S3D1.


Synchrotron configuration: omega and ttheta both rotate about the
transverse axis, so the scattering plane is vertical.  This exploits
the s-polarization and tighter vertical collimation of synchrotron
radiation (Walko 2016).


Default basis: Busing & Levy (1967) — transverse=+x, longitudinal=+y,
vertical=+z.


Sample stack (floor first):
    omega : transverse,   left-handed
    chi   : longitudinal, right-handed
    phi   : transverse,   left-handed


Detector (floor, mechanically independent of sample stack):
    ttheta : transverse, left-handed


omega and ttheta share the same transverse axis; they are
mechanically independent.


References

- W.R. Busing & H.A. Levy, Acta Cryst. 22, 457-464 (1967).

- D.A. Walko, Ref. Module Mater. Sci. Mater. Eng. (2016).

In [ ]:
ahd.factories.list_geometries()

{'fivec': <function ad_hoc_diffractometer.geometry_loader._register_packaged_geometries.<locals>._make_factory.<locals>._factory(**kwargs: 'Any') -> 'AdHocDiffractometer'>,
 'fourch': <function ad_hoc_diffractometer.geometry_loader._register_packaged_geometries.<locals>._make_factory.<locals>._factory(**kwargs: 'Any') -> 'AdHocDiffractometer'>,
 'fourcv': <function ad_hoc_diffractometer.geometry_loader._register_packaged_geometries.<locals>._make_factory.<locals>._factory(**kwargs: 'Any') -> 'AdHocDiffractometer'>,
 'kappa4ch': <function ad_hoc_diffractometer.geometry_loader._register_packaged_geometries.<locals>._make_factory.<locals>._factory(**kwargs: 'Any') -> 'AdHocDiffractometer'>,
 'kappa4cv': <function ad_hoc_diffractometer.geometry_loader._register_packaged_geometries.<locals>._make_factory.<locals>._factory(**kwargs: 'Any') -> 'AdHocDiffractometer'>,
 'kappa6c': <function ad_hoc_diffractometer.geometry_loader._register_packaged_geometries.<locals>._make_factory.<locals>._fact

In [ ]:
ahd.ub_identity(g.sample)

g.summary()
solutions = g.forward(1, 1, 1)
for sol in solutions:
    hkl = g.inverse(sol)
    print(sol, "->", hkl)

AdHocDiffractometer: my_fourcv
  Four-circle Eulerian, vertical scattering plane
  λ = 1.540600 Å  (E = 8.047786 keV)

  Basis vectors:
    vertical     -> +z
    longitudinal -> +y
    transverse   -> +x

  Sample stages (floor first):
    omega     axis=-x    parent=None  angle=0.0 deg
    chi       axis=+y    parent=omega  angle=0.0 deg
    phi       axis=-x    parent=chi  angle=0.0 deg

  Detector stages (floor first):
    ttheta    axis=-x    parent=None  angle=0.0 deg
{'omega': 14.221050819020343, 'chi': 144.73561031724535, 'phi': -45.0, 'ttheta': 28.442101638040686} -> (0.9999999999999998, 0.9999999999999998, 1.0000000000000002)
{'omega': 14.221050819020343, 'chi': 35.264389682754654, 'phi': 135.0, 'ttheta': 28.442101638040686} -> (1.0, 0.9999999999999998, 1.0000000000000002)


In [ ]:
g.wavelength
g.sample
g.sample.lattice
ahd.ub_identity(g.sample)

# Inspect
g.summary()

print("Solutions:")
solutions = g.forward(1, 1, 1)
for sol in solutions:
    print(sol)

for sol in solutions:
    hkl = g.inverse(sol)
    print(f" -> hkl = ({hkl[0]:.4f}, {hkl[1]:.4f}, {hkl[2]:.4f})")

AdHocDiffractometer: my_fourcv
  Four-circle Eulerian, vertical scattering plane
  λ = 1.540600 Å  (E = 8.047786 keV)

  Basis vectors:
    vertical     -> +z
    longitudinal -> +y
    transverse   -> +x

  Sample stages (floor first):
    omega     axis=-x    parent=None  angle=0.0 deg
    chi       axis=+y    parent=omega  angle=0.0 deg
    phi       axis=-x    parent=chi  angle=0.0 deg

  Detector stages (floor first):
    ttheta    axis=-x    parent=None  angle=0.0 deg
Solutions:
{'omega': 14.221050819020343, 'chi': 144.73561031724535, 'phi': -45.0, 'ttheta': 28.442101638040686}
{'omega': 14.221050819020343, 'chi': 35.264389682754654, 'phi': 135.0, 'ttheta': 28.442101638040686}
 -> hkl = (1.0000, 1.0000, 1.0000)
 -> hkl = (1.0000, 1.0000, 1.0000)


In [ ]:
del g

In [ ]:
try:
    ahd.register_geometry_file("./cdi-geometry.yml")
except ValueError:
    pass

In [ ]:
ahd.list_geometries()

{'fivec': <function ad_hoc_diffractometer.geometry_loader._register_packaged_geometries.<locals>._make_factory.<locals>._factory(**kwargs: 'Any') -> 'AdHocDiffractometer'>,
 'fourch': <function ad_hoc_diffractometer.geometry_loader._register_packaged_geometries.<locals>._make_factory.<locals>._factory(**kwargs: 'Any') -> 'AdHocDiffractometer'>,
 'fourcv': <function ad_hoc_diffractometer.geometry_loader._register_packaged_geometries.<locals>._make_factory.<locals>._factory(**kwargs: 'Any') -> 'AdHocDiffractometer'>,
 'kappa4ch': <function ad_hoc_diffractometer.geometry_loader._register_packaged_geometries.<locals>._make_factory.<locals>._factory(**kwargs: 'Any') -> 'AdHocDiffractometer'>,
 'kappa4cv': <function ad_hoc_diffractometer.geometry_loader._register_packaged_geometries.<locals>._make_factory.<locals>._factory(**kwargs: 'Any') -> 'AdHocDiffractometer'>,
 'kappa6c': <function ad_hoc_diffractometer.geometry_loader._register_packaged_geometries.<locals>._make_factory.<locals>._fact

In [ ]:
g = ahd.make_geometry("cdi-geometry")

In [ ]:
g.mode_name = "fixed_alpha"
g.wavelength = 1.5406
g.sample.lattice = ahd.Lattice(a=5.431)
ahd.ub_identity(g.sample)
g.surface_normal = (0, 0, 1)

In [ ]:
g.summary()
solutions = g.forward(1, 1, 1)
for sol in solutions:
    print(sol)
    # hkl = g.inverse(sol)
    # print(sol, "->", hkl)

AdHocDiffractometer: cdi-geometry
  CDI 7-circle diffractometer prototype.

This is a working first model of the CDI stack from beamline notes.
The top two sample rotations are fixed at zero in the initial mode.


The solver has special handling for an Eulerian inner pair named
``chi`` and ``phi``.  These names are solver-facing aliases for the
two lower CDI sample stages noted as omega/chi during beamline
discussion:

    chi  -> gon.sam.c_lg.lrx / omega1 in earlier notes
    phi  -> gon.sam.c_lg.lrz / chi1 in earlier notes


Basis used here: vertical=+Y, longitudinal=+Z, transverse=+X.


Stack (floor first):
    mu                  : vertical
      --> chi           : longitudinal
            --> phi     : transverse
                  --> omega2 : longitudinal, fixed at 0 in fixed_alpha
                        --> chi2 : transverse, fixed at 0 in fixed_alpha
    gamma1              : transverse detector axis, fixed at 0 in fixed_alpha
      --> delta1        : vertical detector axis


In [ ]:
for sol in solutions:
    hkl = g.inverse(sol)
    print(hkl)

(1.0000000000000002, 0.9999999999999997, 1.0)
(0.9999999999999998, 1.0, 1.0)
(0.9999999999999997, 1.0000000000000002, 1.0)
(0.9999999999999998, 1.0000000000000002, 1.0)
(1.0000000000000002, 0.9999999999999998, 0.9999999999999998)
(0.9999999999999998, 1.0000000000000002, 0.9999999999999998)
(1.0, 1.0, 0.9999999999999998)
(1.0000000000000004, 0.9999999999999994, 0.9999999999999998)


In [ ]:
g.mode

ConstraintSet([SampleConstraint('chi2', 0.0), SampleConstraint('omega2', 0.0), ReferenceConstraint('incidence', 5.0), DetectorConstraint('gamma1', 0.0)])

In [ ]:
g.wavelength
E = ahd.radiation.wavelength_to_energy(g.wavelength)
print(E)

lam = ahd.radiation.energy_to_wavelength(E)
print(lam)

8.047786474957826
1.5405999999999997


In [ ]:
### hklpy2
import hklpy2

In [ ]:
fourc = hklpy2.creator(solver="ad_hoc", geometry="fourcv", name="fourc")
fourc.add_sample(name="silicon", a=hklpy2.SI_LATTICE_PARAMETER)
fourc.beam.wavelength.put(1.0)

import math

theta = math.degrees(math.asin(1.0 / (2 * 5.431)))
tth = 2 * theta

r1 = fourc.add_reflection(
    pseudos={"h": 1, "k": 0, "l": 0},
    reals={"omega": theta, "chi": 0, "phi": 0, "ttheta": tth},
    wavelength=1.0,
    name="r1",
)

r2 = fourc.add_reflection(
    pseudos={"h": 0, "k": 1, "l": 0},
    reals={"omega": theta, "chi": 0, "phi": 90, "ttheta": tth},
    wavelength=1.0,
    name="r2",
)

In [ ]:
fourc.core.calc_UB(r1, r2)

[[0.0, 0.0, 1.156906937555034],
 [1.4876207926076293e-16, 1.156906937555034, -7.084011889940698e-17],
 [-1.156906937555034, 2.196021981601699e-16, 7.084011889940695e-17]]

In [ ]:
# ahd.register_geometry_file("./cdi-geometry.yml", name="cdi")

'cdi'

In [ ]:
cdi_diff = hklpy2.creator(name="cdi-geometry", geometry="cdi-geometry", solver="ad_hoc")

In [ ]:
cdi_diff

Hklpy2Diffractometer(prefix='', name='cdi-geometry', settle_time=0.0, timeout=None, egu='', limits=(0, 0), source='computed', read_attrs=['beam', 'beam.wavelength', 'beam.energy', 'h', 'h.readback', 'h.setpoint', 'k', 'k.readback', 'k.setpoint', 'l', 'l.readback', 'l.setpoint', 'mu', 'chi', 'phi', 'omega2', 'chi2', 'gamma1', 'delta1'], configuration_attrs=['beam', 'beam.source_type', 'beam.wavelength_units', 'beam.wavelength_deadband', 'beam.energy_units', 'h', 'k', 'l'], concurrent=True)

In [ ]:
print(hklpy2.__version__)
from hklpy2.user import *

hklpy2.user.set_diffractometer(cdi_diff)

0.7.3


In [ ]:
print(hklpy2.user.pa())
print(hklpy2.user.wh())

diffractometer='cdi-geometry'
AdHocSolver(name='ad_hoc', version='0.11.4', geometry='cdi-geometry')
Sample(name='sample', lattice=Lattice(a=1, system='cubic'))
Orienting reflections: []
U=[[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
UB=[[6.2832, 0.0, 0.0], [0.0, 6.2832, 0.0], [0.0, 0.0, 6.2832]]
constraint: -180.0 <= mu <= 180.0 [cut=-180.0]
constraint: -180.0 <= chi <= 180.0 [cut=-180.0]
constraint: -180.0 <= phi <= 180.0 [cut=-180.0]
constraint: -180.0 <= omega2 <= 180.0 [cut=-180.0]
constraint: -180.0 <= chi2 <= 180.0 [cut=-180.0]
constraint: -180.0 <= gamma1 <= 180.0 [cut=-180.0]
constraint: -180.0 <= delta1 <= 180.0 [cut=-180.0]
Mode: fixed_alpha
beam={'class': 'WavelengthXray', 'source_type': 'Synchrotron X-ray Source', 'energy': 12.3984, 'wavelength': 1.0, 'energy_units': 'keV', 'wavelength_units': 'angstrom'}
pseudos: h=0, k=0, l=0
reals: mu=0, chi=0, phi=0, omega2=0, chi2=0, gamma1=0, delta1=0
extras: incidence=0
None
wavelength=1.0
pseudos: h=0, k=0, l=0
reals: mu=0, c

In [ ]:
hklpy2.user.add_sample("silicon", a=hklpy2.SI_LATTICE_PARAMETER)

Sample(name='silicon', lattice=Lattice(a=5.431, system='cubic'))

In [ ]:
cdi_diff.beam.wavelength.put(1.54)

In [ ]:
# r1 = hklpy2.user.setor(4,0,0, mu=69.0966, omega1=-145.451, chi1=0, omega2=0)
r1 = hklpy2.user.setor(4, 0, 0)
r1

Reflection(name='r_a43b', h=4, k=0, l=0)

In [ ]:
# r2 = hklpy2.user.setor(0,4,0, mu=69.0966, omega1=-145.451, chi1=90, phi=0)
# hklpy2.user.setor?
try:
    r2 = hklpy2.user.setor(1, 1, 1)
except hklpy2.blocks.reflection.ReflectionError:
    pass
r2

Reflection(name='r_e286', h=1, k=1, l=1)

In [ ]:
hklpy2.user.calc_UB(r1, r2)

SolverError: The two reflections are parallel in the phi frame (Q_phi vectors for 'r_a43b' and 'r_e286' are collinear). Choose two reflections that are not parallel.